Реализация подхода классификации документов через создание конечного набора кластеров по эмбеддингам текстов и интерпретация названий с помощью LLM

## Импорт библиотек

In [1]:
pip install -q -U langchain-huggingface langchain-core bertopic bitsandbytes>=0.46.1 natasha optuna gensim

In [2]:
# Базовые библиотеки, утилиты и работа с путями
import os
import re
import gc
import time
import ast
import textwrap
import logging
import warnings
import subprocess
import threading
from typing import List
from tqdm import tqdm

# Обработка данных
import numpy as np
import pandas as pd

# Машинное обучение и предобработка
from sklearn import metrics
from sklearn.metrics import silhouette_score
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import CountVectorizer

# Векторизация, кластеризация и оптимизация
import umap
import optuna
import hdbscan
from hdbscan.validity import validity_index
from sentence_transformers import SentenceTransformer

# Тематическое моделирование (BERTopic)
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance

# Обработка естественного языка (NLP) - Natasha и NLTK
import nltk
from nltk.corpus import stopwords

# Deep Learning, LLM и LangChain
import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Google Drive (для использования в Google Colab)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Функции

In [3]:
def get_topic(text: str) -> str:
    """Определяет краткую тему документа с помощью языковой модели.

    Аргументы:
        text (str): Исходный текст документа (приказ, служебная записка и др.).

    Возвращает:
        str: Выделенная тема документа (2-5 слов в нижнем регистре)
             или сообщение об ошибке/пустом значении.
    """
    if not text or pd.isna(text):
        return "пусто"

    try:
        # Запуск генерации через LangChain
        response = chain.invoke({"document_text": str(text)})

        # Пост-обработка: удаление случайных кавычек, точек и приведение к нижнему регистру
        return response.strip().strip('".').lower()

    except Exception as e:
        return f"ошибка: {e}"

In [4]:
def calculate_clustering_metrics(embeddings: np.ndarray, labels: np.ndarray) -> None:
    """Вычисляет и выводит ключевые метрики качества кластеризации.

    Аргументы:
        embeddings (np.ndarray): Матрица векторных представлений (эмбеддингов).
        labels (np.ndarray): Массив меток кластеров, полученных от алгоритма.
    """
    labels_arr = np.array(labels)

    # Создаем маску для НЕ-шума
    mask = labels_arr >= 0

    clean_embeddings = embeddings[mask]
    clean_labels = labels_arr[mask]

    # Статистика шума
    total_points = len(labels_arr)
    n_noise = np.sum(labels_arr == -1)
    pct_noise = (n_noise / total_points) * 100

    # Уникальные кластеры без шума
    unique_clusters = np.unique(clean_labels)
    n_clusters = len(unique_clusters)

    print(f"Кол-во тем: {n_clusters}")
    print(f"Шум: {n_noise} из {total_points} ({pct_noise:.1f}%)")

    # Проверка условий: минимум 2 кластера и хотя бы по паре точек в них
    if n_clusters > 1 and len(clean_labels) > n_clusters:
        try:
            sil = metrics.silhouette_score(clean_embeddings, clean_labels)
            ch = metrics.calinski_harabasz_score(clean_embeddings, clean_labels)
            db = metrics.davies_bouldin_score(clean_embeddings, clean_labels)

            print(f"Silhouette: {sil:.3f}")
            print(f"Calinski-Harabasz: {ch:.0f}")
            print(f"Davies-Bouldin: {db:.3f}")
        except Exception as e:
            print(f"Ошибка при расчете: {e}")
    else:
        print("Расчет невозможен: недостаточно кластеров или точек в них.")

In [5]:
def clean_noise(text: str) -> str:
    """Очищает текст от технического шума, ссылок, дат, телефонов и тегов.

    Аргументы:
        text (str): Исходная текстовая строка для очистки.

    Возвращает:
        str: Очищенная строка без лишних символов и пробелов.
    """
    if not isinstance(text, str):
        return ""

    # Удаляем URL-адреса и ссылки
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Удаляем Email-адреса
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b", "", text)

    # Удаляем HTML-теги
    text = re.sub(r"<.*?>", "", text)

    # Удаляем даты
    text = re.sub(r"\b\d{2}[-./]\d{2}[-./]\d{4}\b", "", text)
    text = re.sub(r"\b\d{4}[-./]\d{2}[-./]\d{2}\b", "", text)

    # Удаляем номера телефонов
    text = re.sub(
        r"\+?\d{1,3}[-.\s]?\(?\d{1,4}\)?[-.\s]?\d{1,4}[-.\s]?\d{1,4}[-.\s]?\d{1,9}",
        "",
        text,
    )

    # Заменяем b2b-разделители на пробелы
    text = re.sub(r"[_—–─]{2,}", " ", text)

    # Удаляем повторяющиеся знаки препинания
    text = re.sub(r"([!?.])\1+", r"\1", text)

    # Схлопываем множественные пробелы и переносы строк
    text = re.sub(r"\s+", " ", text)

    return text.strip().lower()

In [6]:
# Инициализация компонентов Natasha
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,
    Doc
)

segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)

# Инициализация компонентов Natasha
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)

def clean_single_text_lemma(text: str, stop_words) -> str:
    """
    Лемматизирует и очищает ОДИН текст (строку).
    Возвращает очищенную строку с леммами через пробел.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    text_only_letters = re.sub(r'[^а-яА-Яa-zA-Z\s]', ' ', text)
    doc = Doc(text_only_letters)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)

    doc_lemmas = []
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
        lemma = token.lemma.lower()

        # Фильтрация шума (поиск по set происходит мгновенно)
        if (lemma not in stop_words
                and token.pos != "PUNCT"
                and not lemma.isdigit()
                and len(lemma) > 2):
            doc_lemmas.append(lemma)

    # Возвращаем склеенную строку лемм
    return " ".join(doc_lemmas) if doc_lemmas else ""

In [7]:
def objective(trial: optuna.Trial, embed_norm: np.ndarray) -> float:
    n_neighbors = trial.suggest_categorical("n_neighbors", [5, 10, 20, 30, 50, 70, 100])
    n_components = trial.suggest_categorical("n_components", [3, 5, 10, 15, 20, 30, 50])
    min_dist = trial.suggest_float("min_dist", 0.0, 0.1)

    min_cluster_size = trial.suggest_categorical("min_cluster_size", [5, 10, 20, 30, 50, 70])
    min_samples = trial.suggest_categorical("min_samples", [2, 3, 5, 10, 15, 25])

    try:
        # Обучаем UMAP
        umap_embeddings = umap.UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            min_dist=min_dist,
            random_state=42
        ).fit_transform(embed_norm)

        # Обучаем HDBSCAN
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='leaf',
            prediction_data=True
        ).fit(umap_embeddings)

        labels = clusterer.labels_
        mask = labels != -1
        n_clusters = len(set(labels[mask]))
        outlier_perc = np.mean(labels == -1)

        # проверка на минимальное количество кластеров
        if n_clusters < 2:
            print(f"Trial {trial.number}: [Пропущен] Мало кластеров ({n_clusters})")
            return -1.0

        # защита от избыточного шума (более 75%)
        if outlier_perc > 0.75:
            print(f"Trial {trial.number}: [Пропущен] Слишком много шума ({outlier_perc:.1%})")
            return -1.0

        # считаем на отфильтрованных ядрах кластеров
        score = silhouette_score(umap_embeddings[mask], labels[mask])

        # Лог процесса
        print(f"Trial {trial.number}: Silhouette={score:.4f} | Кластеров={n_clusters} | Шум={outlier_perc:.1%} | NN={n_neighbors}, MC={min_cluster_size}")

        return score

    except Exception as e:
        print(f"Trial {trial.number}: Ошибка во время выполнения -> {e}")
        return -1.0

## Загрузка данных

In [8]:
df = pd.read_csv('/content/drive/MyDrive/TextClust/texts.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             1000 non-null   int64 
 1   document_text  1000 non-null   object
dtypes: int64(1), object(1)
memory usage: 15.8+ KB


## Получение эмбеддингов

In [ ]:
embed_model = SentenceTransformer('BAAI/bge-m3')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
# Пакетная генерация эмбеддингов
texts_to_encode = [item for item in df['document_text_preproc']]
embeddings = embed_model.encode(
    texts_to_encode,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [ ]:
# Создаем DataFrame из двух колонок
df_output = pd.DataFrame({'document_text_preproc': texts_to_encode})
df_output['embedding'] = list(embeddings)

df_output.to_feather('/content/drive/MyDrive/TextClust/document_texts_and_embeddings.feather')

print(f"Сохранено чанков: {len(df_output)}")

Сохранено чанков: 1000


In [ ]:
pd.set_option('display.max_colwidth', 500)
df_output

,document_text_preproc,embedding
0,"ПРИКАЗ №101. О проведении внепланового ТО станков ЧПУ в цехе №3. В связи с отклонением точности обработки, ПРИКАЗЫВАЮ: 1. Главному инженеру организовать проверку геометрии станков до 15.10. 2. Начальнику цеха приостановить работы на участках №2 и №4.","[-0.024791837, -0.008809981, -0.013596547, -0.0020596932, -0.020105492, -0.032709755, -0.03583683, 0.017604059, -0.013942341, 0.032927442, -0.017972875, -0.02394282, 0.0385142, 0.0037623923, -0.019265994, -0.043239534, 0.029281631, 0.007617364, 0.0029799596, -0.0008202229, 0.0061981915, -0.0018954976, 0.025457792, 0.0014714539, -0.024934076, 0.010001191, -0.013051579, -0.0058462746, 0.050871648, 0.018979715, -0.02704507, -0.018037869, 0.024075605, -0.03866371, -0.017455598, -0.039170735, 0.0..."
1,"СЛУЖЕБНАЯ ЗАПИСКА. Директору по закупкам. От нач. литейного участка. Прошу рассмотреть закупку 5 тонн чугуна СЧ20. Текущие запасы на исходе, что ставит под угрозу выполнение плана по отливке корпусов для 'МашПром'.","[0.00026988104, 0.02324181, -0.047127638, 0.0047974004, -0.006357464, -0.037006665, -0.02510175, 0.036362324, -0.0047525023, 0.0015306971, 0.027848557, -0.011987951, -0.012083356, -0.015384819, 0.0023335877, -0.058105413, -0.005835626, 0.0029501314, -0.050506715, -0.0014358809, 0.0042777746, -0.007868346, 0.05451023, 0.043815803, -0.021150054, -0.0053996677, -0.025642576, 0.012222371, 0.027984174, 0.032095723, 0.007170146, -0.017716972, 0.031525277, -0.007808531, 0.0064484808, -0.037471116, ..."
2,ИСХОДЯЩЕЕ ПИСЬМО №45/02. Руководителю ООО 'МеталлСнаб'. Уведомляем о задержке оплаты по договору №12-89 от 01.09. в связи с техническим сбоем в банке. Оплата будет произведена до конца недели. Приносим извинения.,"[-0.036639217, 0.019446326, -0.035344254, -0.004799226, -0.023842953, -0.002299977, -0.033047162, 0.015177227, -0.029074738, 0.006326412, 0.045594625, 0.009180854, 0.0028942917, 0.024371816, -0.025913268, -0.032655828, 0.0014478104, 0.0021966381, -0.002089358, 0.01224427, -0.0041722693, -0.03373229, 0.040767673, 0.007405502, 0.014572736, -0.014850345, 0.023232624, -0.013868447, 0.05055326, -0.002001114, 0.010834112, -0.00842928, 0.0003294532, -0.019455975, -0.0015054455, -0.028307091, 0.0458..."
3,"ПРИКАЗ №112. Об утверждении графика отпусков на ремонтный период. Для обеспечения работы инструментального цеха при ремонте кровли, ПРИКАЗЫВАЮ: 1. Утвердить перенос отпусков мастеров согласно приложению №1.","[-0.053496122, 0.006899469, -0.035935044, 0.0058075776, -0.019114403, -0.053256657, -0.007882538, 0.029617267, -0.010462525, -0.018215464, 0.008899451, 0.008989221, -0.017162712, 0.03549331, 0.034829397, -0.016748292, 0.05209779, 0.023153406, 0.012273234, -0.0059251464, -0.009869383, -0.036776595, 0.020597365, 0.054361243, -0.049253736, 0.01536284, -0.017487047, 0.013740337, 0.04136652, 0.0094967885, -0.0022771743, 0.0053266683, 0.018641403, 0.019776557, -0.005798334, -0.019487726, -0.009277..."
4,СЛУЖЕБНАЯ ЗАПИСКА. Главному технологу. От инженера ОТК. При приемке партии валов (чертеж ) выявлен брак по шероховатости. Прошу провести аудит техпроцесса шлиф-обработки и проверить состояние кругов.,"[-0.03445354, 0.032488752, -0.046638764, 0.0052144728, 0.014366459, -0.03712058, -0.011221448, 0.023017941, -0.019441973, 0.0018942987, 0.049564544, -0.042440876, 0.054303125, 0.007074009, 0.041578192, -0.021058558, 0.024661824, 0.019617673, -0.022267276, 0.03460966, -0.0036528907, 0.033365607, 0.036243048, -0.04357314, -0.016295146, -0.018852739, -0.018849272, 0.0034545027, 0.037076347, 0.0008628612, 0.020199979, -0.010118406, -0.01403662, -0.034476615, -0.057125684, -0.06481441, 0.05614689..."
...,...,...
995,ПРИКАЗ №2090. О проведении аттестации рабочих мест по условиям труда на новом участке ЧПУ. Организовать замеры уровня вибрации и освещенности. Срок исполнения — до 25.12. Ответственный — инж. по ОТ.,"[-0.017633798, -0.013361762, -0.024568891, -0.024850959, -0.013994071, -0.03329191, -0.024194539, -0.011773128, -0.0066081467, 0.012612093,

## BERTopic

In [10]:
df = pd.read_feather('/content/drive/MyDrive/TextClust/document_texts_and_embeddings.feather')

In [11]:
embed=np.stack(df['embedding'].values)
embed.shape

(1000, 1024)

In [12]:
# L2 нормализация для евклидового пространства
embed_norm = normalize(embed)

In [13]:
nltk.download('stopwords')
stop_words = stopwords.words('russian')+['гост', 'ост', 'ту', 'рис', 'табл', 'рисунок', 'таблица', 'пункт', 'тз', 'кд', 'тд', 'нтд', 'гост', 'ост', 'ниокр', 'нии', 'кб',
    'отк', 'ппр', 'то', 'тр', 'кр', 'зип', 'рэ', 'пс', 'рис', 'табл', 'см', 'мм', 'км', 'кг', 'квт', 'мпа', 'шт', 'об', 'мин', 'сек', 'оао', 'пао', 'ао', 'тоо', 'ооо', 'зао', 'фгуп', 'гуп', 'ип', 'гк', 'нп',
    'нии', 'кб', 'цкб', 'пки', 'ран', 'ниу', 'втуз', 'нпо', 'по', 'пп', 'хк','минпромторг', 'минэнерго', 'минобороны', 'мчс', 'ростех', 'росатом',
    'роскосмос', 'ржд', 'росстандарт', 'фстэк', 'записка', 'служебная', 'служебные', 'приказ', 'приказы', 'письмо', 'письма',
    'исходящее', 'исходящие', 'входящее', 'входящие', 'необходимость', 'проведение',
    'касательно', 'повод', 'поводу', 'относительно', 'справка', 'акт', 'акты',
    'протокол', 'протоколы', 'заявка', 'заявки', 'отчет', 'отчеты', 'записка']

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [14]:
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    stop_words=stop_words,
    token_pattern=r"(?u)\b[а-яА-ЯёЁ]{2,}\b",
)

In [15]:
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=False,
    bm25_weighting=True
)

In [ ]:
# Инициализация и запуск оптимизации
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: objective(trial, embed_norm), n_trials=50)

print("\nЛучшие параметры:", study.best_params)

[I 2026-05-29 19:11:22,528] A new study created in memory with name: no-name-3274dddc-b917-4fbd-b58a-7a7ea00ef567
[I 2026-05-29 19:11:28,715] Trial 0 finished with value: 0.6415014863014221 and parameters: {'n_neighbors': 10, 'n_components': 10, 'min_dist': 0.09075832272334992, 'min_cluster_size': 50, 'min_samples': 25}. Best is trial 0 with value: 0.6415014863014221.


Trial 0: Silhouette=0.6415 | Кластеров=5 | Шум=19.0% | NN=10, MC=50


[I 2026-05-29 19:11:44,455] Trial 1 finished with value: 0.4407379925251007 and parameters: {'n_neighbors': 100, 'n_components': 15, 'min_dist': 0.09230271874752657, 'min_cluster_size': 10, 'min_samples': 3}. Best is trial 0 with value: 0.6415014863014221.


Trial 1: Silhouette=0.4407 | Кластеров=39 | Шум=14.5% | NN=100, MC=10


[I 2026-05-29 19:11:55,666] Trial 2 finished with value: 0.5091317296028137 and parameters: {'n_neighbors': 30, 'n_components': 15, 'min_dist': 0.016860883223587097, 'min_cluster_size': 50, 'min_samples': 2}. Best is trial 0 with value: 0.6415014863014221.


...

Trial 47: Silhouette=0.5609 | Кластеров=21 | Шум=8.5% | NN=30, MC=20


[I 2026-05-29 19:15:28,798] Trial 48 finished with value: 0.5795047283172607 and parameters: {'n_neighbors': 70, 'n_components': 15, 'min_dist': 0.019179033462440154, 'min_cluster_size': 5, 'min_samples': 10}. Best is trial 23 with value: 0.7687163949012756.


Trial 48: Silhouette=0.5795 | Кластеров=33 | Шум=32.6% | NN=70, MC=5


[I 2026-05-29 19:15:31,848] Trial 49 finished with value: 0.5679279565811157 and parameters: {'n_neighbors': 5, 'n_components': 3, 'min_dist': 0.03590600695351351, 'min_cluster_size': 30, 'min_samples': 5}. Best is trial 23 with value: 0.7687163949012756.


Trial 49: Silhouette=0.5679 | Кластеров=16 | Шум=13.2% | NN=5, MC=30

Лучшие параметры: {'n_neighbors': 5, 'n_components': 20, 'min_dist': 0.019360479975685907, 'min_cluster_size': 5, 'min_samples': 10}


In [ ]:
optuna.visualization.plot_optimization_history(study)

In [16]:
umap_model = umap.UMAP(
    n_neighbors=5,
    n_components=20,
    min_dist=0.019,
    metric='euclidean',
    random_state=42
)

In [17]:
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=10,
    gen_min_span_tree=True,
    metric='euclidean',
    cluster_selection_method='eom',
)

In [18]:
representation_model = {"MMR": MaximalMarginalRelevance(diversity=0.5)}

In [19]:
topic_model = BERTopic(
    language='russian',
    embedding_model=None,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model
)

In [20]:
topic_model.save("/content/drive/MyDrive/TextClust/my_model", serialization="safetensors")

TypeError: 'NoneType' object is not iterable

In [ ]:
topic_model = BERTopic.load("/content/drive/MyDrive/TextClust/my_model")

2026-05-29 19:49:49,433 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


In [21]:
topics, _ = topic_model.fit_transform(df['document_text_preproc'].astype(str).tolist(), embeddings=embed_norm)

In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,MMR,Representative_Docs
0,-1,110,-1_проверке_комиссии_труда_письмо,"[проверке, комиссии, труда, письмо, исходящее, исходящее письмо, комиссии проверке, назначении комиссии, письмо директору, подшипников]","[проверке, комиссии, труда, письмо, исходящее, исходящее письмо, комиссии проверке, назначении комиссии, письмо директору, подшипников, случае, состояния, приказ назначении, назначении, мест, рабочих мест, путей, приказ, просим, уровня, цехе, директору, условиям, лимиты, аттестации рабочих, мест условиям, условиям труда, рабочих, организовать замеры, аттестации]","[ПРИКАЗ №1690. О проведении аттестации рабочих мест по условиям труда. Организовать замеры уровня вибрации на участках прессового оборудования цеха №1. Срок исполнения — до 15.12. Ответственный — инж. по ОТ., ПРИКАЗ №890. О проведении аттестации рабочих мест по условиям труда. Организовать замеры уровня вибрации на участках прессового оборудования. Срок исполнения — до 15.12. Ответственный — инж. по ОТ., ПРИКАЗ №1290. О проведении аттестации рабочих мест по условиям труда. Организовать замер..."
1,0,78,0_нач омтс_омтс_закупка_нач,"[нач омтс, омтс, закупка, нач, требуется закупка, главному бухгалтеру, бухгалтеру, бухгалтеру нач, омтс прошу, срочно требуется]","[нач омтс, омтс, закупка, нач, требуется закупка, главному бухгалтеру, бухгалтеру, бухгалтеру нач, омтс прошу, срочно требуется, срочно, прошу выделить, выделить, служебная записка, служебная, записка, требуется, средства, омтс нач, литров, прошу, участка, выделить денежные, денежные средства, денежные, нач транспортного, транспортного, отдел снабжения, записка главному, финансовому]","[СЛУЖЕБНАЯ ЗАПИСКА. Главному бухгалтеру. От нач. ОМТС. Прошу выделить денежные средства на оплату счета №112 за приобретение 100 литров СОЖ марки Blaser для работы на высокоскоростных станках ЧПУ., СЛУЖЕБНАЯ ЗАПИСКА. Главному бухгалтеру. От нач. ОМТС. Прошу выделить денежные средства в размере руб. на приобретение измерительного инструмента (микрометры, штангенциркули) для нужд цеха №1., СЛУЖЕБНАЯ ЗАПИСКА. Главному бухгалтеру. От нач. ОМТС. Прошу выделить денежные средства на оплату счета №9..."
2,1,44,1_директору ит_ит нач_ит_записка директору,"[директору ит, ит нач, ит, записка директору, директору, бухгалтерии, пдо прошу, прошу, нач бухгалтерии, бухгалтерии прошу]","[директору ит, ит нач, ит, записка директору, директору, бухгалтерии, пдо прошу, прошу, нач бухгалтерии, бухгалтерии прошу, нач пдо, пдо, настроить, прошу настроить, доступ, системе, прошу предоставить, служебная записка, служебная, записка, нач, прошу приобрести, отдела, приобрести, дополнительный модуль, модуль, кадров прошу, автоматизации, дополнительный, ведущему]","[СЛУЖЕБНАЯ ЗАПИСКА. Директору по ИТ. От нач. отдела кадров. Прошу приобрести дополнительный модуль 'Подбор персонала' для системы 1С в целях автоматизации процесса обработки резюме кандидатов., СЛУЖЕБНАЯ ЗАПИСКА. Директору по ИТ. От нач. отдела кадров. Прошу приобрести дополнительный модуль 'Подбор персонала' для системы 1С в целях автоматизации процесса обработки резюме., СЛУЖЕБНАЯ ЗАПИСКА. Директору по ИТ. От нач. отдела кадров. Прошу приобрести дополнительный модуль 'Подбор персонала' для..."
3,2,41,2_назначить_ответственным_приказ назначении_назначении,"[назначить, ответственным, приказ назначении, назначении, ответственного, назначении ответственного, исправное, исправное состояние, состояние, ответственного исправное]","[назначить, ответственным, приказ назначении, назначении, ответственного, назначении ответственного, исправное, исправное состояние, состояние, ответственного исправное, назначить мастера, назначении ответственных, приказ закреплении, закреплении, ответственных, систем, закрепить, эксплуатацию, механика, приказ, службой, содержание, мастера, состояние систем, ответственным проверку, ответственной, своевременную, ответственных содержание, ведение, проверку]","[ПРИКАЗ №1805. О назначении ответственных за содержание и эксплуатацию систем вентиляци

In [ ]:
# метрики
calculate_clustering_metrics(topic_model.umap_model.transform(embed_norm),topics)
print(f"DBCV: {(topic_model.hdbscan_model.relative_validity_):.2f}")

Кол-во тем: 38
Шум: 110 из 1000 (11.0%)
Silhouette: 0.751
Calinski-Harabasz: 5094
Davies-Bouldin: 0.341
DBCV: 0.41


In [ ]:
# используем n_components=2, чтобы можно было нарисовать точки на плоскости
reduced_embeddings_2d = umap.UMAP(
    n_neighbors=15,
    n_components=2,
    metric='euclidean',
    random_state=42
).fit_transform(embed)

topic_model.visualize_documents(df['document_text_preproc'], reduced_embeddings=reduced_embeddings_2d, hide_annotations=True)

In [22]:
# распределяем шум
new_topics = topic_model.reduce_outliers(
    documents=df['document_text_preproc'].tolist(),
    topics=topics,
    strategy="embeddings", # явно указываем стратегию по эмбеддингам
    embeddings=embed_norm,
    threshold=0.6
)
topic_model.update_topics(df['document_text_preproc'].tolist(), topics=new_topics, top_n_words=10)

2026-05-30 10:56:24,627 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [ ]:
# визуализация после распределения
topic_model.visualize_documents(df['document_text_preproc'], reduced_embeddings=reduced_embeddings_2d, hide_annotations=True)

In [ ]:
# метрики после распределения шума
calculate_clustering_metrics(topic_model.umap_model.transform(embed_norm),new_topics)
print(f"DBCV: {(topic_model.hdbscan_model.relative_validity_):.2f}")

Кол-во тем: 38
Шум: 0 из 1000 (0.0%)
Silhouette: 0.539
Calinski-Harabasz: 577
Davies-Bouldin: 1.055
DBCV: 0.41


In [23]:
topic_info = topic_model.get_topic_info()
topic_info.to_csv("/content/drive/MyDrive/TextClust/topic_info_mmr.csv", index=False)

## Получение тем LLM

In [3]:
topic_info = pd.read_csv("/content/drive/MyDrive/TextClust/topic_info_mmr.csv")
topic_model = BERTopic.load("/content/drive/MyDrive/TextClust/my_model")

2026-05-30 11:47:42,127 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


In [5]:
# Очистка памяти
gc.collect()
torch.cuda.empty_cache()

model_id = "Qwen/Qwen2.5-14B"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},              # Жесткая посадка на GPU:0
    low_cpu_mem_usage=True,          # Экономия RAM при инициализации
    trust_remote_code=True
)

config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/47.5k [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=20,
    max_length=None,       # Явно сбрасываем max_length, чтобы убрать варнинг
    return_full_text=False,
    do_sample=True,      # Включаем сэмплирование (обязательно для работы temperature)
    temperature=0.3,     # Оптимально для строгого следования инструкциям
    top_p=0.9,
)

In [ ]:
llm = HuggingFacePipeline(
    pipeline=pipe,
    pipeline_kwargs={
        "tokenizer": tokenizer,
        # Добавляем перенос строки, чтобы модель не генерировала лишний текст
        "stop_strings": ["<|im_end|>", "<|endoftext|>", "\n"]
    }
)

In [ ]:
template = """<|im_start|>system
Ты — эксперт по документообороту в машиностроении. Твоя задача — изучить ключевые слова документов завода и назвать тему.

Правила:
1. Пиши ТОЛЬКО название темы (1, 2 или максимум 3 слова).
2. Используй аббревиатуры (ОМТС, ПДО, ИТ, ЧС, ОТ и ТБ).
3. Никаких вводных слов, никаких глаголов и фраз вроде "приказ о...", "записка о...".
4. Пиши строго существительными в именителном падеже (например: "Материально-техническое снабжение", "Входной контроль").

Примеры:
Ключевые слова: график, дежурство, начальник, цех, сменность
Тема: График дежурств персонала

Ключевые слова: склад, металл, закупка, поставщик, снабжение
Тема: Материально-техническое снабжение
<|im_end|>
<|im_start|>user
Назови краткую тему (1-3 слова) для ключевых слов: {keywords}
<|im_end|>
<|im_start|>assistant
Тема: """


prompt = PromptTemplate.from_template(template)
chain = prompt | llm

In [22]:
topic_info = topic_model.get_topic_info()

# Словарь для хранения результатов: {ID_кластера: "сгенерированная_тема"}
cluster_themes = {}

for index, row in topic_info.iterrows():
    topic_id = row['Topic']

    # Пропускаем выбросы/шум
    if topic_id == -1:
        cluster_themes[topic_id] = "выбросы (шум)"
        continue

    # Извлекаем список слов MMR
    mmr_keywords_list = row['Representation']

    # Очищаем и лемматизируем список с помощью Natasha
    filtered_list = mmr_keywords_list

    # Объединяем чистые леммы
    keywords_string = ", ".join(filtered_list)

    # Запуск генерации через LangChain цепочку
    try:
        generated_theme = chain.invoke({"keywords": keywords_string})

        # Очищаем от возможных пробелов по краям
        clean_theme = generated_theme.strip().strip('."\'`').lower()
        cluster_themes[topic_id] = clean_theme

        print(f"Кластер {topic_id}: {clean_theme}")

    except Exception as e:
        print(f"Ошибка при обработке кластера {topic_id}: {e}")
        cluster_themes[topic_id] = "ошибка генерации"

# Добавление сгенерированных тем обратно в датафрейм BERTopic
topic_info['Generated_Theme'] = topic_info['Topic'].map(cluster_themes)

Кластер 0: закупка, срочно требуется закуп заготовок, срочно требуется закуп заготовок, срочно требуется закуп
Кластер 1: , записка, записка, записка, записка, 'записка'
assistant:
Кластер 2: , исправное состояние, исправное состояние, исправное состояние, ответственного, исправное состояние,


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Кластер 3: , срок, срок, срок, срок, срок, срок, срок, срок, срок, срок
Кластер 4: из строя, выход из строя, выход из строя, выход из строя, выход из машиностро
Кластер 5: приказу, согласно приказу, согласно приказу, согласно приказу, согласно приказу, согласно


KeyboardInterrupt: 

In [ ]:

# Извлекаем топ-слова для всех топиков (исключая шум -1) из результатов MMR
all_topics = topic_model.get_topics()
topic_words = [
    [word for word, _ in all_topics[topic]] 
    for topic in all_topics if topic != -1
]

# Полная токенизация и лемматизация всего корпуса документов
cleaned_corpus_strings = clean_text_lemma(all_docs_list)
tokenized_corpus = [doc.lower().split() for doc in cleaned_corpus_strings]

# Создаем структуру словаря Gensim
dictionary = Dictionary(tokenized_corpus)


# coherence score
coherence_model = CoherenceModel(
    topics=topic_words, 
    texts=tokenized_corpus, 
    dictionary=dictionary, 
    coherence='c_v'
)
coherence_score = coherence_model.get_coherence()
print(f"Topic Coherence (C_V): {coherence_score:.4f}")


# topic diversity
def calculate_topic_diversity(topics, topk=10):
    all_words = []
    for topic in topics:
        all_words.extend(topic[:topk])
    if not all_words:
        return 0.0
    return len(set(all_words)) / len(all_words)

diversity_score = calculate_topic_diversity(topic_words, topk=10)
print(f"Topic Diversity (Top-10): {diversity_score:.4f}")

In [ ]:
topic_info

In [ ]:
topic_info.to_csv('/content/drive/MyDrive/TextClust/df_with_topics.csv', index=False, encoding='utf-8-sig')